In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install transformers accelerate bitsandbytes sentencepiece faiss-cpu sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 102.5 MB/s eta 0:00:00


In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


In [ ]:
import os
import re
import json
from pathlib import Path

import torch
import pandas as pd
import numpy as np
import faiss

from sentence_transformers import (
    SentenceTransformer,
    CrossEncoder
)

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

In [ ]:
PROJECT_PATH = Path(
    "/content/drive/MyDrive/uterine-emg-rag"
)

EMBEDDINGS_PATH = PROJECT_PATH / "embeddings"
FAISS_PATH = PROJECT_PATH / "faiss_index"
EVALUATION_PATH = PROJECT_PATH / "evaluation"

print(PROJECT_PATH)

/content/drive/MyDrive/uterine-emg-rag


In [ ]:
metadata = pd.read_csv(
    EMBEDDINGS_PATH / "chunk_metadata.csv"
)

print("Number of chunks:", len(metadata))

metadata.head()

Number of chunks: 833


,paper,chunk_id,text,characters
0,paper1,0,Contents lists available at ScienceDirect\nArt...,927
1,paper1,1,Keywords:\nUterine electromyography\nUterine a...,946
2,paper1,2,"transform, and for data classification, such a...",996
3,paper1,3,3\n2.1. \nUterine contraction classification ....,831
4,paper1,4,6\n3.1.1. \nSensing electrodes ..................,966


In [ ]:
index = faiss.read_index(
    str(FAISS_PATH / "uterine_emg.index")
)

print("FAISS index loaded")
print("Number of vectors:", index.ntotal)

FAISS index loaded
Number of vectors: 833


In [ ]:
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

In [ ]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

In [ ]:
llm = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quant_config,
    device_map="auto"
)

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

In [ ]:
messages = [
    {
        "role": "user",
        "content": "What is uterine electromyography?"
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(llm.device)

with torch.no_grad():

    outputs = llm.generate(
        **inputs,
        max_new_tokens=100,
        temperature=0.2,
        do_sample=True
    )

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(response)

Uterine electromyography (EMG) is not a standard medical procedure or diagnostic tool. It's possible you might be confusing it with another related concept. Here’s a clarification:

1. **Electromyography (EMG)**: This is a diagnostic test that measures the electrical activity of muscles and nerves. EMG can be used to diagnose various conditions such as muscle disorders, nerve damage, or other neurological issues.

2. **Uterine Electromyography (UEMG


In [ ]:
def retrieve_candidates(
    query,
    embedding_model,
    index,
    metadata,
    k=20
):

    query_embedding = embedding_model.encode(
        [query]
    ).astype("float32")

    faiss.normalize_L2(query_embedding)

    scores, indices = index.search(
        query_embedding,
        k
    )

    results = []

    for rank, idx in enumerate(
        indices[0],
        start=1
    ):

        if idx < 0:
            continue

        results.append({
            "faiss_rank": rank,
            "faiss_score": float(
                scores[0][rank - 1]
            ),
            "index": int(idx),
            "paper": metadata.iloc[idx]["paper"],
            "chunk_id": metadata.iloc[idx]["chunk_id"],
            "text": metadata.iloc[idx]["text"]
        })

    return results

In [ ]:
def retrieve_and_rerank(
    query,
    embedding_model,
    index,
    metadata,
    reranker,
    candidate_k=20,
    final_k=5
):

    candidates = retrieve_candidates(
        query=query,
        embedding_model=embedding_model,
        index=index,
        metadata=metadata,
        k=candidate_k
    )

    pairs = [
        [query, result["text"]]
        for result in candidates
    ]

    scores = reranker.predict(pairs)

    for result, score in zip(
        candidates,
        scores
    ):

        result["rerank_score"] = float(score)

    candidates = sorted(
        candidates,
        key=lambda x: x["rerank_score"],
        reverse=True
    )

    for rank, result in enumerate(
        candidates,
        start=1
    ):

        result["rerank_rank"] = rank

    return candidates[:final_k]

In [ ]:
query = "What are the characteristics of uterine EMG signals?"

results = retrieve_and_rerank(
    query=query,
    embedding_model=embedding_model,
    index=index,
    metadata=metadata,
    reranker=reranker,
    candidate_k=20,
    final_k=5
)

In [ ]:
for result in results:

    print("=" * 80)

    print(
        "Rank:",
        result["rerank_rank"]
    )

    print(
        "Paper:",
        result["paper"]
    )

    print(
        "Chunk:",
        result["chunk_id"]
    )

    print(
        "FAISS score:",
        result["faiss_score"]
    )

    print(
        "Rerank score:",
        result["rerank_score"]
    )

    print()

    print(
        result["text"][:1000]
    )

Rank: 1
Paper: paper2
Chunk: 25
FAISS score: 0.7202328443527222
Rerank score: 8.740561485290527

home environments.
2.4. Signal Characteristics and Typical EHG Profiles
Uterine EMG signals have distinctive characteristics in both the time and frequency do-
mains. Key features include amplitude, frequency content, temporal pattern, and propagation.
Amplitude: The EHG is a low-amplitude signal. During uterine contractions, the peak-
to-peak amplitude typically ranges from a few tens of microvolts up to about 1 millivolt on
the abdominal surface. Studies report that the higher-frequency “fast” component of EHG
usually has amplitude under 1 mV, whereas the low-frequency “slow” baseline shift can
reach several millivolts [10]. As labor progresses, the EHG amplitude tends to increase due
to stronger and more synchronized contractions. However, absolute amplitude can vary
widely between patients and with electrode placement, so amplitude alone is not a specific
indicator of labor unless norma

In [ ]:
def build_citation_context(results):

    context_parts = []

    for i, result in enumerate(
        results,
        start=1
    ):

        context_parts.append(
            f"""
[Source {i}]

Paper: {result["paper"]}
Chunk ID: {result["chunk_id"]}

Evidence:
{result["text"]}
"""
        )

    return "\n".join(context_parts)

In [ ]:
def build_citation_prompt(
    query,
    context
):

    prompt = f"""
You are a scientific research assistant specializing
in uterine electromyography (EMG/EHG).

Answer the user's question using ONLY the research
evidence provided below.

IMPORTANT RULES:

1. Do not use outside knowledge.

2. Do not invent facts.

3. Every factual claim must be supported by at least
   one source from the provided context.

4. After each factual statement or group of closely
   related statements, provide a citation using the
   format:

   [Source N]

5. Only cite sources that actually support the claim.

6. Do not cite a source merely because it is related
   to the topic.

7. If multiple sources support a statement, you may cite:

   [Source 1][Source 3]

8. If the retrieved context does not contain enough
   information, say:

   "I don't have enough information in the retrieved
   research papers to answer this question."

9. Do not create citations such as [Source 6] if only
   five sources were provided.

10. Keep the answer scientifically precise and concise.

RESEARCH CONTEXT
================

{context}

================

USER QUESTION
=============

{query}

=============

ANSWER
======
"""

    return prompt

In [ ]:
def generate_answer(
    query,
    context,
    max_new_tokens=300
):

    prompt = build_citation_prompt(
        query,
        context
    )

    messages = [
        {
            "role": "system",
            "content": (
                "You are a scientific research assistant. "
                "Use only the supplied research context."
            )
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(llm.device)

    with torch.no_grad():

        outputs = llm.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.2,
            do_sample=True
        )

    answer = tokenizer.decode(
        outputs[0][
            inputs["input_ids"].shape[1]:
        ],
        skip_special_tokens=True
    )

    return answer

In [ ]:
def extract_citations(answer):

    citations = re.findall(
        r"\[Source\s+(\d+)\]",
        answer
    )

    return sorted(
        set(
            int(c)
            for c in citations
        )
    )

In [ ]:
def validate_citations(
    answer,
    num_sources
):

    citations = extract_citations(
        answer
    )

    invalid = [
        citation
        for citation in citations
        if citation < 1 or citation > num_sources
    ]

    return {
        "citations": citations,
        "invalid_citations": invalid,
        "all_citations_valid": len(invalid) == 0
    }

In [ ]:
def format_source_details(results):

    formatted = []

    for i, result in enumerate(
        results,
        start=1
    ):

        formatted.append({
            "source_id": i,
            "paper": result["paper"],
            "chunk_id": result["chunk_id"],
            "faiss_rank": result["faiss_rank"],
            "faiss_score": result["faiss_score"],
            "rerank_rank": result["rerank_rank"],
            "rerank_score": result["rerank_score"],
            "text": result["text"]
        })

    return formatted

In [ ]:
def citation_aware_rag(
    query,
    candidate_k=20,
    final_k=5,
    max_new_tokens=300
):

    # ------------------------------------------
    # 1. RETRIEVAL + RERANKING
    # ------------------------------------------

    results = retrieve_and_rerank(
        query=query,
        embedding_model=embedding_model,
        index=index,
        metadata=metadata,
        reranker=reranker,
        candidate_k=candidate_k,
        final_k=final_k
    )

    # ------------------------------------------
    # 2. BUILD CITATION CONTEXT
    # ------------------------------------------

    context = build_citation_context(
        results
    )

    # ------------------------------------------
    # 3. GENERATE ANSWER
    # ------------------------------------------

    answer = generate_answer(
        query=query,
        context=context,
        max_new_tokens=max_new_tokens
    )

    # ------------------------------------------
    # 4. VALIDATE CITATIONS
    # ------------------------------------------

    citation_info = validate_citations(
        answer=answer,
        num_sources=len(results)
    )

    # ------------------------------------------
    # 5. FORMAT SOURCES
    # ------------------------------------------

    sources = format_source_details(
        results
    )

    return {
        "query": query,
        "answer": answer,
        "citations": citation_info,
        "sources": sources
    }

In [ ]:
result = citation_aware_rag(
    "What are the characteristics of uterine EMG signals?"
)

print(result["answer"])

Uterine EMG signals exhibit distinct characteristics in both the time and frequency domains. Key features include amplitude, frequency content, temporal pattern, and propagation. 

The amplitude of uterine EMG signals ranges from a few tens of microvolts to about 1 millivolt on the abdominal surface during uterine contractions. Specifically, the high-frequency "fast" component usually has an amplitude under 1 mV, while the low-frequency "slow" baseline shift can reach several millivolts. As labor progresses, the amplitude of uterine EMG signals tends to increase due to stronger and more synchronized contractions. However, absolute amplitude can vary widely between patients and with electrode placement, making it not a specific indicator of labor unless normalized within-subjects [Source 1].

In terms of frequency content, uterine EMG signals show a composite of slow baseline shifts and fast-wave components up to approximately 1 Hz. The signals are typically characterized by slow, synch

In [ ]:
print(
    "Citations:",
    result["citations"]
)

Citations: {'citations': [1, 5], 'invalid_citations': [], 'all_citations_valid': True}


In [ ]:
for source in result["sources"]:

    print("=" * 80)

    print(
        f"[Source {source['source_id']}]"
    )

    print(
        "Paper:",
        source["paper"]
    )

    print(
        "Chunk:",
        source["chunk_id"]
    )

    print(
        "Rerank score:",
        source["rerank_score"]
    )

[Source 1]
Paper: paper2
Chunk: 25
Rerank score: 8.740561485290527
[Source 2]
Paper: paper2
Chunk: 55
Rerank score: 8.26572036743164
[Source 3]
Paper: paper2
Chunk: 44
Rerank score: 6.40878963470459
[Source 4]
Paper: paper2
Chunk: 54
Rerank score: 5.959493160247803
[Source 5]
Paper: paper2
Chunk: 87
Rerank score: 5.5933732986450195


In [ ]:
def display_rag_result(result):

    print("\n" + "=" * 80)
    print("ANSWER")
    print("=" * 80)

    print(result["answer"])

    print("\n" + "=" * 80)
    print("CITATION VALIDATION")
    print("=" * 80)

    print(
        "Citations:",
        result["citations"]["citations"]
    )

    print(
        "Invalid citations:",
        result["citations"]["invalid_citations"]
    )

    print(
        "All citations valid:",
        result["citations"]["all_citations_valid"]
    )

    print("\n" + "=" * 80)
    print("SOURCES")
    print("=" * 80)

    for source in result["sources"]:

        print(
            f"\n[Source {source['source_id']}]"
        )

        print(
            "Paper:",
            source["paper"]
        )

        print(
            "Chunk:",
            source["chunk_id"]
        )

        print(
            "FAISS rank:",
            source["faiss_rank"]
        )

        print(
            "Rerank rank:",
            source["rerank_rank"]
        )

In [ ]:
display_rag_result(result)


ANSWER
Uterine EMG signals exhibit distinct characteristics in both the time and frequency domains. Key features include amplitude, frequency content, temporal pattern, and propagation. 

The amplitude of uterine EMG signals ranges from a few tens of microvolts to about 1 millivolt on the abdominal surface during uterine contractions. Specifically, the high-frequency "fast" component usually has an amplitude under 1 mV, while the low-frequency "slow" baseline shift can reach several millivolts. As labor progresses, the amplitude of uterine EMG signals tends to increase due to stronger and more synchronized contractions. However, absolute amplitude can vary widely between patients and with electrode placement, making it not a specific indicator of labor unless normalized within-subjects [Source 1].

In terms of frequency content, uterine EMG signals show a composite of slow baseline shifts and fast-wave components up to approximately 1 Hz. The signals are typically characterized by slo

In [ ]:
test_questions = [
    "What are the characteristics of uterine EMG signals?",

    "What frequency ranges are commonly observed in uterine EMG?",

    "How is uterine EMG used for detecting labor?",

    "What factors affect uterine EMG signal quality?",

    "What signal processing methods are used for uterine EMG?",

    "How does uterine electrical activity change during pregnancy?",

    "What are the challenges associated with uterine EMG analysis?"
]

In [ ]:
all_results = []

for question in test_questions:

    print("\n")
    print("=" * 100)
    print(question)
    print("=" * 100)

    result = citation_aware_rag(
        question
    )

    print(result["answer"])

    print(
        "\nCitations:",
        result["citations"]["citations"]
    )

    all_results.append(
        result
    )



What are the characteristics of uterine EMG signals?
Uterine EMG signals exhibit distinctive characteristics in both the time and frequency domains. Key features include amplitude, frequency content, temporal pattern, and propagation. 

The amplitude of uterine EMG signals ranges from a few tens of microvolts up to about 1 millivolt on the abdominal surface during uterine contractions. Specifically, the higher-frequency "fast" component usually has an amplitude under 1 mV, while the low-frequency "slow" baseline shift can reach several millivolts. As labor progresses, the amplitude of uterine EMG signals tends to increase due to stronger and more synchronized contractions. However, absolute amplitude can vary widely between patients and with electrode placement, making it not a specific indicator of labor unless normalized within-subjects [Source 1].

In terms of frequency content, uterine EMG signals show a composite of slow baseline shifts and fast-wave components up to approximate

In [ ]:
evaluation_rows = []

for result in all_results:

    evaluation_rows.append({

        "query": result["query"],

        "answer": result["answer"],

        "num_citations": len(
            result["citations"]["citations"]
        ),

        "num_invalid_citations": len(
            result["citations"]["invalid_citations"]
        ),

        "all_citations_valid":
            result["citations"]["all_citations_valid"]
    })

evaluation_df = pd.DataFrame(
    evaluation_rows
)

evaluation_df

,query,answer,num_citations,num_invalid_citations,all_citations_valid
0,What are the characteristics of uterine EMG si...,Uterine EMG signals exhibit distinctive charac...,2,0,True
1,What frequency ranges are commonly observed in...,Commonly observed frequency ranges in uterine ...,3,0,True
2,How is uterine EMG used for detecting labor?,Uterine EMG (Electromyogram) is utilized for d...,2,0,True
3,What factors affect uterine EMG signal quality?,The quality of uterine EMG signals is affected...,0,0,True
4,What signal processing methods are used for ut...,Signal processing methods for uterine EMG incl...,2,0,True
5,How does uterine electrical activity change du...,"During pregnancy, uterine electrical activity ...",4,1,False
6,What are the challenges associated with uterin...,The challenges associated with uterine EMG ana...,3,0,True


In [ ]:
citation_validity_rate = (
    evaluation_df["all_citations_valid"]
    .mean()
)

print(
    f"Citation validity rate: "
    f"{citation_validity_rate:.2%}"
)

Citation validity rate: 85.71%


In [ ]:
EVALUATION_PATH.mkdir(
    parents=True,
    exist_ok=True
)

evaluation_df.to_csv(
    EVALUATION_PATH /
    "citation_evaluation.csv",
    index=False
)

In [ ]:
detailed_results = []

for result in all_results:

    detailed_results.append({
        "query": result["query"],
        "answer": result["answer"],
        "citations": result["citations"],
        "sources": result["sources"]
    })

In [ ]:
with open(
    EVALUATION_PATH / "citation_rag_results.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        detailed_results,
        f,
        indent=2,
        ensure_ascii=False,
        default=lambda x: x.item() if hasattr(x, "item") else str(x)
    )

In [ ]:
json_path = (
    EVALUATION_PATH /
    "citation_rag_results.json"
)

print("Saved to:", json_path)
print("File exists:", json_path.exists())
print("File size:", json_path.stat().st_size, "bytes")

Saved to: /content/drive/MyDrive/uterine-emg-rag/evaluation/citation_rag_results.json
File exists: True
File size: 52419 bytes


In [ ]:
with open(
    EVALUATION_PATH / "citation_rag_results.json",
    "r",
    encoding="utf-8"
) as f:

    test_data = json.load(f)

print("JSON loaded successfully")
print("Number of results:", len(test_data))

JSON loaded successfully
Number of results: 7
